# Explore: your GPT, thinking out loud

This notebook shows what you built actually *doing* things: the causal mask as a
picture, a real attention map on real Shakespeare, and the same model sampled again and
again as it trains - from noise to speech. It *consumes* your `data.py` /
`attention.py` / `model.py` / `train.py`; no solutions live here. If a cell fails, its
milestone isn't done yet - `python check.py` is the source of truth.

Each section says which milestone it needs. Run cells top to bottom. The training cell
in section 4 takes a couple of minutes - it's training a GPT, after all.

*Needs:* `pip install jupyter matplotlib` (on top of torch).

In [ ]:
%load_ext autoreload
%autoreload 2

import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

from data import load_text, build_vocab, split_data, get_batch
from attention import causal_weights, xbow_loop
from model import BigramLanguageModel, GPT
from train import train_gpt, estimate_loss

text = load_text()
stoi, itos, encode, decode = build_vocab(text)
vocab_size = len(stoi)
train_data, val_data = split_data(text, encode)
print(f"{len(text):,} characters, vocab {vocab_size}")
print(text[:180])

## 1. What one batch row really is  *(needs Milestone 1)*

One row of x packs block_size separate prediction problems - from one character of
context up to the full window. This is why your model can start generating from a
single newline: it trained on short contexts too.

In [ ]:
x, y = get_batch(train_data, block_size=8, batch_size=1,
                 generator=torch.Generator().manual_seed(1337))
row_x, row_y = x[0], y[0]
print("the chunk:", repr(decode(torch.cat([row_x, row_y[-1:]]).tolist())))
print()
for t in range(8):
    ctx = decode(row_x[:t + 1].tolist())
    print(f"  given {ctx!r:>12} -> predict {decode([row_y[t].item()])!r}")

## 2. The trick, as a picture  *(needs Milestone 3)*

The whole mechanism of causal attention in one matrix: row t is position t's mixing
recipe over the past. Uniform for now - a trained head's only upgrade is making these
weights data-dependent.

In [ ]:
wei = causal_weights(8)
fig, ax = plt.subplots(figsize=(5, 5))
im = ax.imshow(wei, cmap="Blues")
for i in range(8):
    for j in range(8):
        if wei[i, j] > 0:
            ax.text(j, i, f"{wei[i, j]:.2f}", ha="center", va="center",
                    fontsize=8, color="black")
ax.set_xlabel("attends to position")
ax.set_ylabel("position")
ax.set_title("causal_weights(8): each row averages its past")
plt.show()

xr = torch.randn(2, 8, 4, generator=torch.Generator().manual_seed(1))
print("wei @ x equals your loop:", torch.allclose(wei @ xr, xbow_loop(xr), atol=1e-5))

## 3. The bar: the bigram  *(needs Milestones 2 & 8)*

Module 2's model, trained properly - one character of context, doing its best. Keep its
loss curve and its prose in mind; they're what attention is about to beat.

In [ ]:
torch.manual_seed(1337)
bigram = BigramLanguageModel(vocab_size)
bigram_lossi = train_gpt(bigram, train_data, block_size=8, batch_size=32,
                         steps=2000, lr=1e-2,
                         generator=torch.Generator().manual_seed(1337))
b_val = estimate_loss(bigram, train_data, val_data, 8, 32, eval_iters=50,
                      generator=torch.Generator().manual_seed(7))["val"]
print(f"bigram val loss: {b_val:.4f}")
bigram.eval()
ctx = torch.tensor([[stoi["\n"]]], dtype=torch.long)
out = bigram.generate(ctx, 150, generator=torch.Generator().manual_seed(42))
print("bigram prose:", repr(decode(out[0].tolist())))

## 4. Watch it learn to speak  *(needs Milestones 7 & 8)*

The same GPT, the same prompt, sampled at four points during training. This cell is the
slow one (~2 minutes) - you are watching gradient descent acquire language.

In [ ]:
torch.manual_seed(1337)
gpt = GPT(vocab_size, block_size=32, n_embd=48, n_head=4, n_layer=2)
print(f"a pocket GPT: {sum(p.numel() for p in gpt.parameters()):,} parameters")

g = torch.Generator().manual_seed(1337)
gpt_lossi = []
checkpoints = [0, 300, 700, 1200]  # cumulative steps
prompt = torch.tensor([[stoi["\n"]]], dtype=torch.long)
for prev, now in zip(checkpoints, checkpoints[1:] + [None]):
    gpt.eval()
    sample = gpt.generate(prompt, 120, generator=torch.Generator().manual_seed(42))
    print(f"--- after {prev} steps " + "-" * 34)
    print(decode(sample[0].tolist()).strip("\n")[:120])
    if now is not None:
        gpt_lossi += train_gpt(gpt, train_data, block_size=32, batch_size=16,
                               steps=now - prev, generator=g)

In [ ]:
val = estimate_loss(gpt, train_data, val_data, 32, 16, eval_iters=50,
                    generator=torch.Generator().manual_seed(7))["val"]
plt.figure(figsize=(10, 3.5))
plt.plot(torch.tensor(bigram_lossi).view(-1, 50).mean(1), label=f"bigram (val {b_val:.2f})")
plt.plot(torch.tensor(gpt_lossi).view(-1, 50).mean(1), label=f"pocket GPT (val {val:.2f})")
plt.xlabel("step / 50")
plt.ylabel("loss (mean of 50)")
plt.legend()
plt.title("one character of context vs thirty-two, with attention")
plt.show()
print(f"pocket GPT val: {val:.4f} - and train.py's model is twice this size")

## 5. What attention actually looks at  *(needs Milestone 4, uses the model above)*

A real attention map: head 0 of block 0, on a real line of Shakespeare. Each row is a
character deciding how much of its past to read. The tests proved the triangle; here
you can see the *choices* inside it.

In [ ]:
snippet = "First Citizen:"
idx = torch.tensor([encode(snippet)], dtype=torch.long)
T = idx.shape[1]

gpt.eval()
with torch.no_grad():
    x = gpt.token_embedding_table(idx) + gpt.position_embedding_table(torch.arange(T))
    xln = gpt.blocks[0].ln1(x)
    h = gpt.blocks[0].sa.heads[0]
    q, k = h.query(xln), h.key(xln)
    wei = q @ k.transpose(-2, -1) * k.shape[-1] ** -0.5
    wei = wei.masked_fill(torch.tril(torch.ones(T, T)) == 0, float("-inf"))
    wei = F.softmax(wei, dim=-1)

fig, ax = plt.subplots(figsize=(6.5, 6))
ax.imshow(wei[0], cmap="Blues")
ax.set_xticks(range(T)); ax.set_xticklabels(list(snippet))
ax.set_yticks(range(T)); ax.set_yticklabels(list(snippet))
ax.set_xlabel("attends to")
ax.set_title("block 0, head 0, on a real line")
plt.show()

Every row sums to 1, nothing above the diagonal - your Milestone 4 tests, visible.
The dark columns are characters this head finds broadly useful; other heads (try
`heads[1]`, or block 1) have different tastes. In a 96-layer GPT it's this exact
picture, thousands of times over.

For the full experience run `python train.py` - the payoff model is bigger, trains
longer, and its verse is noticeably better. Next lecture:
[Let's build the GPT Tokenizer](https://www.youtube.com/watch?v=zduSFxRajkE) - why real
GPTs don't read characters at all.